# Experiment 11 — Rollout fidelity

**Paper reference.** §3 of v15 — the linear policy needs to support multi-step rollouts to be useful for §4 (counter-strategy design).

**Goal.** Take the chosen linear seller model and simulate **complete negotiations** against each of the four buyer schemes A–D. Compare the simulated trajectory statistics (mean ask per round, deal-rate, final-price distribution) to the real data.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

def fit_rollout_model(scheme, policy):
    fam, mdl = picked(scheme, policy)
    sub = turns.query('scheme == @scheme and policy == @policy')
    meta, X, y, _ = nv.build_design_matrix(sub, fam)
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_n, y_n)
    cols = nv.FEATURE_SPECS[fam]['cols']
    return est, fam, cols


### Scheme samplers (mirror Appendix B.1 §B)

In [ ]:
def buyer_step(scheme, B_prev, S_prev, B_init, M, v, t, T, lam, rng):
    if scheme == 'A':                       # Anchored 'IID'
        return max(B_prev, rng.uniform(0, S_prev))
    if scheme == 'B':                       # Slow fractional concession
        return B_prev + (S_prev - B_prev) * rng.beta(1, 3)
    if scheme == 'C':                       # Random walk with retreats
        step = rng.normal(0.04, np.sqrt(0.01))   # paper says variance=0.01 -> std=0.1
        return max(0.0, min(S_prev, B_prev + (M - v) * step))
    if scheme == 'D':                       # Perturbed pacing
        return max(0.0, min(S_prev,
                            (t/T) * M * lam + (M - v) * rng.uniform(-0.08, 0.08)))
    raise ValueError(scheme)


def empirical_S1(raw, scheme, policy, vehicle_id):
    """Sample a realistic first ask from the real data for this (scheme,policy,vehicle)."""
    s1 = (raw.query('scheme == @scheme and policy == @policy and '
                    'vehicle_id == @vehicle_id and round == 1')['seller_price_raw']
              .dropna().values)
    return s1


def simulate_one(est, fam, cols, scheme, M, v, T, S1_bank, rng=None):
    """Closed-loop simulation. S1 is bootstrapped from the real distribution
    (S1_bank: 1d array of empirical first-asks)."""
    rng = rng or np.random.default_rng()
    margin = M - v
    B = float(rng.uniform(0, 0.3 * M))
    # Bootstrap S_1 from real data (or fall back to M)
    S = float(rng.choice(S1_bank)) if len(S1_bank) > 0 else float(M)
    S = float(np.clip(S, v, M))
    history = [{'t': 1, 'B': B, 'S': S}]
    lam = float(rng.uniform(0.7, 1.3))
    Y_prev = 0.0
    B_prev = B; S_prev = S
    for t in range(2, T+1):
        B_new = buyer_step(scheme, B_prev, S_prev, history[0]['B'], M, v, t, T, lam, rng)
        S_lag1 = float(np.clip((S_prev - v)/margin, nv.ETA, 1-nv.ETA))
        B_norm = (B_new - v) / margin
        # rough lag2 fallback (use lag1 again on the second iteration)
        if t == 2:
            S_tilde_L_lag2 = np.log(S_lag1/(1-S_lag1))
            B_tilde_lag2   = (B - v)/margin
        feat_row = {
            'B_tilde':         B_norm,
            'S_tilde_lag1':    S_lag1,
            'S_tilde_L_lag1':  np.log(S_lag1/(1-S_lag1)),
            'B_tilde_lag1':    (B_prev - v) / margin,
            'B_tilde_lag1_pos':max((B_prev - v) / margin, 0.0),
            'B_tilde_lag1_neg':max(-(B_prev - v) / margin, 0.0),
            'B_tilde_pos':     max(B_norm, 0.0),
            'B_tilde_neg':     max(-B_norm, 0.0),
            'dB':              B_norm - (B_prev - v) / margin,
            'dB_pos':          max(B_norm - (B_prev - v) / margin, 0.0),
            'dB_neg':          max(-(B_norm - (B_prev - v) / margin), 0.0),
            'Y_lag1':          Y_prev,
            'B_tilde_lag2':    B_tilde_lag2 if t == 2 else (B_prev - v)/margin,
            'S_tilde_L_lag2':  S_tilde_L_lag2 if t == 2 else np.log(S_lag1/(1-S_lag1)),
        }
        xrow = np.array([[feat_row[c] for c in cols]])
        Y_pred = float(est.predict(xrow)[0])
        S_tilde_new = nv.predicted_S_tilde_from_Y(np.array([Y_pred]), np.array([S_lag1]))[0]
        S_new = float(S_tilde_new) * margin + v
        # agreement / floor check
        if S_new <= 1.005 * B_new or S_new <= v:
            history.append({'t': t, 'B': B_new, 'S': max(S_new, v)})
            return history, True
        history.append({'t': t, 'B': B_new, 'S': S_new})
        Y_prev = Y_pred
        B_prev, S_prev = B_new, S_new
    return history, False


### Run simulations and compare to real data

In [ ]:
FOCUS = 'A'   # which buyer scheme's seller-fitted model to roll out
POL   = 'PVD'

# Pick one vehicle (or do all 5)
VID = 1
car = raw.query('scheme == @FOCUS and policy == @POL and vehicle_id == @VID').iloc[0]
M, v = float(car['M']), float(car['v'])
print(f'Simulating {POL} (linear from scheme {FOCUS}) against all 4 buyer schemes; vehicle {VID}: M={M}, v={v}')

est, fam, cols = fit_rollout_model(FOCUS, POL)
print(f'  using family {fam}')

rng = np.random.default_rng(0)
sim_runs = {}
for sch in SCHEMES:
    S1_bank = empirical_S1(raw, FOCUS, POL, VID)
    print(f'  bootstrap S1 bank for {sch}: n={len(S1_bank)}, mean=${{S1_bank.mean():.0f}}' if len(S1_bank) else f'  no S1 data for {sch}')
    runs = [simulate_one(est, fam, cols, sch, M, v, T=12, S1_bank=S1_bank, rng=rng) for _ in range(50)]
    sim_runs[sch] = runs
    deal = np.mean([agreed for _, agreed in runs])
    print(f'  buyer scheme {sch}: simulated deal-rate = {deal:.2%}')

### Compare per-round mean ask: real vs simulated

In [ ]:
def per_round_mean_ask(real_or_sim, key='S'):
    if isinstance(real_or_sim, pd.DataFrame):
        return real_or_sim.groupby('round')[key].mean()
    rows = []
    for hist, _ in real_or_sim:
        for h in hist:
            rows.append({'round': h['t'], 'S': h['S'], 'B': h['B']})
    df = pd.DataFrame(rows)
    return df.groupby('round')[key].mean()

real_subset = (raw.query('scheme == @FOCUS and policy == @POL and vehicle_id == @VID')
               .rename(columns={'seller_price_raw':'S','buyer_price_raw':'B'}))
# Truncate seller to [v, M] for fair comparison
real_subset['S'] = real_subset['S'].clip(lower=v, upper=M)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
real_curve = per_round_mean_ask(real_subset)
for ax, sch in zip(axes, SCHEMES):
    sim_curve = per_round_mean_ask(sim_runs[sch])
    ax.plot(real_curve.index, real_curve.values, 'o-', label=f'real (scheme {FOCUS})')
    ax.plot(sim_curve.index, sim_curve.values, 's--', label=f'sim (scheme {sch})')
    ax.set_title(f'against scheme {sch}')
    ax.set_xlabel('round'); ax.legend(fontsize=8)
axes[0].set_ylabel('mean ask ($)')
fig.suptitle(f'Per-round mean ask — {POL} (linear fit from scheme {FOCUS})')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp11_rollout.png'))
plt.show()

### What this tells you

- The **trajectory** of the simulated seller should track the real seller's mean ask shape on its own scheme (column `FOCUS`). If it doesn't, the chosen linear model is failing at one-step prediction *or* compounding errors over many steps. The OOF MAE in Exp03 gives the one-step picture; this notebook gives the closed-loop picture.
- The **off-scheme columns** test whether the seller's behaviour is scheme-invariant in practice: a scheme-invariant linear seller should produce a similar curve across all four panels. Differences point to scheme-specific behaviour the linear model didn't catch.

This rollout is also the building block for §4: once you have a fitted seller policy, optimal counterplay is a planning problem against this learned dynamics.